# CLRP — paper-faithful reproduction (Gu, Yang, Tresp 2018, Fig. 3)

Class-discriminative LRP on multi-object scenes from *Understanding Individual Decisions of CNNs via Contrastive Backpropagation* (ACCV 2018). Plain LRP is NOT class-discriminative — both class targets produce nearly identical heatmaps. CLRP isolates the regions that contributed to the target class above and beyond their contribution to the rest.

Setup:
- Model: torchvision VGG-16 (ImageNet weights).
- Targets: class 340 (zebra) and class 386 (African elephant).
- Config: z+ everywhere with the z^B input rule via the `input_conv` fact
  (`rule={**BASE, 'input_conv': ('zbox', {...}), 'ConvolutionBackward': 'zplus', 'AddmmBackward': 'zplus'}`) — paper's recipe.

In [ ]:
import sys
sys.path.insert(0, '..')          # examples/showcase: _common
sys.path.insert(0, '../../..')    # repo root: autoLRP (or `pip install -e .`)
from pathlib import Path
import numpy as np
import torch
from torchvision.models import vgg16, VGG16_Weights
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt

import autoLRP as autolrp
from autoLRP import LRPConfig, BASE
import _common  # noqa

device = 'cuda' if torch.cuda.is_available() else 'cpu'
ZEBRA, ELEPHANT = 340, 386
DATA_DIR = Path('../../../data/clrp_paper')
IMAGES = ['ze1', 'ze2', 'ze3', 'ze4']

In [ ]:
model = vgg16(weights=VGG16_Weights.DEFAULT).eval().to(device)
preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                          [0.229, 0.224, 0.225]),
])
MEAN = np.array([0.485, 0.456, 0.406])
STD  = np.array([0.229, 0.224, 0.225])

CFG = LRPConfig(rule={**BASE,
                      'input_conv':          ('zbox', {'low': -1.0, 'high': 1.0}),   # z^B at the input conv
                      'ConvolutionBackward': 'zplus',
                      'AddmmBackward':       'zplus'})                               # z+ everywhere else

## Side-by-side: plain LRP vs `autolrp.clrp`

Each row shows one image with three columns per target class: original, plain LRP, and CLRP. Plain LRP fires on both objects for either target; CLRP isolates the target object.

In [ ]:
def lrp_plain(model, img, target_idx, cfg):
    x = autolrp.tensor(img.clone())
    out = model(x)
    out[0, int(target_idx)].lrp(config=cfg)
    return x.relevance.detach()

def to_rgb(R):
    """max(0, R/R.max()) per pixel, paper's `visualize` recipe."""
    rgb = R[0].permute(1, 2, 0).cpu().numpy()
    rgb = np.maximum(0, rgb)
    m = rgb.max() + 1e-12
    return rgb / m

In [ ]:
n_rows = len(IMAGES)
panel = 2.2
fig = plt.figure(figsize=(5 * panel, n_rows * panel + 0.6),
                  facecolor='black')
gs = fig.add_gridspec(n_rows + 1, 5,
                       hspace=0.06, wspace=0.06,
                       height_ratios=[0.16] + [1.0] * n_rows)

headers = ['Original',
           'LRP\nzebra', 'CLRP\nzebra',
           'LRP\nelephant', 'CLRP\nelephant']
for c, t in enumerate(headers):
    ax = fig.add_subplot(gs[0, c])
    ax.text(0.5, 0.3, t, ha='center', va='center',
            color='white', fontsize=12, fontweight='bold')
    ax.set_facecolor('black'); ax.axis('off')

for row, name in enumerate(IMAGES):
    img_pil = Image.open(DATA_DIR / f'{name}.jpg').convert('RGB')
    img = preprocess(img_pil).unsqueeze(0).to(device)

    R_lrp_z  = lrp_plain(model, img, ZEBRA,    CFG)
    R_lrp_e  = lrp_plain(model, img, ELEPHANT, CFG)
    R_clrp_z = autolrp.clrp(model, img, ZEBRA,    config=CFG)
    R_clrp_e = autolrp.clrp(model, img, ELEPHANT, config=CFG)

    panels = [
        np.array(img_pil.resize((224, 224))) / 255.0,
        to_rgb(R_lrp_z),  to_rgb(R_clrp_z),
        to_rgb(R_lrp_e),  to_rgb(R_clrp_e),
    ]
    for c, p in enumerate(panels):
        ax = fig.add_subplot(gs[row + 1, c])
        ax.imshow(p); ax.set_facecolor('black'); ax.axis('off')

plt.show()